In [2]:
import os
import gc
import re
import time
import sqlite3
import requests 
import numpy as np
import pandas as pd
import urllib.request
from bs4 import BeautifulSoup

In [3]:
def get_soup(url):
    '''
    Create a soup object of a web url

    Args: 
        url - String (ex. https://basketball-reference.com/)

    Returns:
        soup - BeautifulSoup object
    '''
    try:
        with urllib.request.urlopen(url) as response:
            html = response.read()
    except urllib.error.URLError as e:
        print(f"Error fetching URL: {e.reason}")

    ### create the soup object
    soup = BeautifulSoup(html, 'html.parser')

    return soup

https://www.basketball-reference.com/

Teams:

Eastern Conference:
- ATL — Atlanta Hawks
- BOS — Boston Celtics
- BRK — Brooklyn Nets
- CHI — Chicago Bulls
- CHO — Charlotte Hornets
- CLE — Cleveland Cavaliers
- DET — Detroit Pistons
- IND — Indiana Pacers
- MIA — Miami Heat
- MIL — Milwaukee Bucks
- NYK — New York Knicks
- ORL — Orlando Magic
- PHI — Philadelphia 76ers
- TOR — Toronto Raptors
- WAS — Washington Wizards

Western Conference:
- DAL — Dallas Mavericks
- DEN — Denver Nuggets
- GSW — Golden State Warriors
- HOU — Houston Rockets
- LAC — Los Angeles Clippers
- LAL — Los Angeles Lakers
- MEM — Memphis Grizzlies
- MIN — Minnesota Timberwolves
- NOP — New Orleans Pelicans
- OKC — Oklahoma City Thunder
- PHO — Phoenix Suns
- POR — Portland Trail Blazers
- SAC — Sacramento Kings
- SAS — San Antonio Spurs
- UTA — Utah Jazz


Players:
- Basketball-Reference encodes player names as: /(last initial)/(first 5 letters of last name)(first 2 letters of first name)01.html
  - i.e. `j/jamesle01.html`
  - Names with the same last initial and first name will increment that last digits of the encoded name as they entered into the BR database.

Functions:

`get_player_ids`
> This function is used to find the unique player ID Basketball-Reference.com uses for each player on a specific team.
- inputs: 
  - team - [string] three letter abbreviation from Basketball-Reference.com of team name (ex. SAC)
  - year - [int] playing year (ex. 2025-2026 season then 2026) 
- output: 
  - [dictionary] 
    - keys -> player name
    - values -> Basketball-Reference.com ID (ex. 'DeMar DeRozan': 'derozde01')


`get_player_data`
> Pulls all possible tables from a specific player's player history and inserts data into Sqlite3 database
- inputs:
  - username: [sting] unique Basketball-Reference.com player ID from `get_player_ids`
- outputs:
  - [dictionary] 
    - keys -> name of table
    - values -> shape of table

`get_seasonal_stats`
> Pulls player game-to-game data for each season (regular and/or post) and inserts data into Sqlite3 database
- inputs:
  - username: [string] unique Basketball-Reference.com player ID from `get_player_ids`
- outputs:
  - [dictionary] 
    - keys -> name of table
    - values -> shape of table



### Player ID Scraping
- Create a function which pulls the player ID Basketball-Reference associates to each player from their team data.

In [4]:
def get_player_ids(team, year):
    url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'

    soup = get_soup(url)

    ids = []
    players = {}

    rows = soup.find("tbody").find_all("tr")

    for row in rows:
        player_cell = row.find("td", {"data-stat": "player"})
        
        if player_cell:
            link = player_cell.find("a")
            
            if link:
                name = link.text
                href = link.get("href")
                split = href.split('/')[-1]
                if '.html' in split:
                    split = split[0:split.find('.')]
                    players[name] = split
                else:
                    players[name] = href
                    print('Check Player List')

    if not players:
        print('No Team Found')
    else:
        # print(ids)
        return players
    
kings = get_player_ids('SAC', 2026)
kings

{'DeMar DeRozan': 'derozde01',
 'Nique Clifford': 'cliffni01',
 'Maxime Raynaud': 'raynama01',
 'Precious Achiuwa': 'achiupr01',
 'Russell Westbrook': 'westbru01',
 'Malik Monk': 'monkma01',
 'Dylan Cardwell': 'cardwdy01',
 'Drew Eubanks': 'eubandr01',
 'Zach LaVine': 'lavinza01',
 'Devin Carter': 'cartede02',
 'Daeqwon Plowden': 'plowdda01',
 'Doug McDermott': 'mcderdo01',
 'Keegan Murray': 'murrake02',
 'Killian Hayes': 'hayeski01',
 'Domantas Sabonis': 'sabondo01',
 'Patrick Baldwin Jr.': 'baldwpa01',
 'Isaiah Stevens': 'steveis01',
 "De'Andre Hunter": 'huntede01'}

In [5]:
def get_player_data(username):
    url = f"https://www.basketball-reference.com/players/{username[0]}/{username}.html"

    soup = get_soup(url)
    tbodies = soup.find_all("tbody")

    tables = {}

    for i, tbody in enumerate(tbodies):
        rows_data = []
        table_name = None

        for row in tbody.find_all("tr"):
            row_id = row.get("id")

            if row_id and table_name is None:
                table_name = row_id.split(".")[0]

            cells = row.find_all(["th", "td"])

            row_dict = {}
            for j, cell in enumerate(cells):
                col_name = cell.get("data-stat")
                text = cell.get_text(strip=True)

                if col_name is None:
                    col_name = f"col_{j}"

                row_dict[col_name] = text

            if row_dict:
                rows_data.append(row_dict)

        df = pd.DataFrame(rows_data)

        df = df.replace("", pd.NA)
        df = df.dropna(thresh=df.shape[1] / 2)

        df["player"] = username

        if table_name is None:
            table_name = f"table_{i}"

            if table_name == "table_0" and df.shape[0] == 5:
                table_name = "past_5_games"
            elif table_name == "table_3" and df.shape[0] == 7:
                table_name = "stathead_insight"

        tables[table_name] = df

    ## For Testing May Remove
    for name, df in tables.items():
        print(f"{name}: {df.shape}")

    ### Insert to Database+

    with sqlite3.connect("example.db", timeout=60) as conn:

        for key, df in tables.items():

            sql_table_name = f"{key}"

            # append if table exists, create if not
            df.to_sql(
                sql_table_name,
                conn,
                if_exists="append",
                index=False
            )

            # remove duplicate rows
            deduped_df = pd.read_sql_query(
                f'SELECT DISTINCT * FROM "{sql_table_name}"',
                conn
            )

            deduped_df.to_sql(
                sql_table_name,
                conn,
                if_exists="replace",
                index=False
            )

            print(
                f"{sql_table_name}: "
                f"{len(df)} inserted, "
                f"{len(deduped_df)} total after dedupe"
            )

    return tables

data = get_player_data("westbru01")

past_5_games: (5, 28)
per_game_stats: (20, 32)
per_game_stats_post: (14, 32)
stathead_insight: (7, 4)
totals_stats: (20, 33)
totals_stats_post: (14, 33)
per_minute_stats: (20, 32)
per_minute_stats_post: (14, 32)
advanced: (20, 30)
advanced_post: (14, 30)
shooting: (20, 32)
shooting_post: (14, 32)
past_5_games: 5 inserted, 10 total after dedupe
per_game_stats: 20 inserted, 25 total after dedupe
per_game_stats_post: 14 inserted, 17 total after dedupe
stathead_insight: 7 inserted, 14 total after dedupe
totals_stats: 20 inserted, 25 total after dedupe
totals_stats_post: 14 inserted, 17 total after dedupe
per_minute_stats: 20 inserted, 25 total after dedupe
per_minute_stats_post: 14 inserted, 17 total after dedupe
advanced: 20 inserted, 25 total after dedupe
advanced_post: 14 inserted, 17 total after dedupe
shooting: 20 inserted, 25 total after dedupe
shooting_post: 14 inserted, 17 total after dedupe


In [6]:
def get_seasonal_stats(player_id):
    table_name = f"{player_id}_per_game_stats"

    with sqlite3.connect("example.db") as conn:
        season_df = pd.read_sql_query(
            f"SELECT * FROM {table_name}",
            conn
        )

    player_years = (
        season_df["year_id"]
        .dropna()
        .astype(str)
        .str.split("-")
        .str[0]
        .astype(int)
        .add(1)
        .astype(str)
        .unique()
    )

    tables = {}

    for year in player_years:
        link = f"https://www.basketball-reference.com/players/{player_id[0]}/{player_id}/gamelog/{year}/"

        soup = get_soup(link)
        tbodies = soup.find_all("tbody")

        for i, tbody in enumerate(tbodies):
            rows_data = []
            table_name = None

            for row in tbody.find_all("tr"):
                # skip repeated header rows
                if "thead" in row.get("class", []):
                    continue

                row_id = row.get("id")

                if row_id and table_name is None:
                    table_name = row_id.split(".")[0]

                cells = row.find_all(["th", "td"])

                row_dict = {}
                for j, cell in enumerate(cells):
                    col_name = cell.get("data-stat")

                    if col_name is None:
                        col_name = f"col_{j}"

                    row_dict[col_name] = cell.get_text(strip=True)

                if row_dict:
                    rows_data.append(row_dict)

            df = pd.DataFrame(rows_data)

            if df.empty:
                continue

            df = df.replace("", pd.NA)

            # Keep rows where at least half the columns are filled
            min_non_empty = int(df.shape[1] / 2)
            df = df.dropna(thresh=min_non_empty)

            df["player"] = player_id
            df["year"] = year

            if table_name is None:
                table_name = f"table_{i}"

            # prevents overwriting seasons
            key = f"{year}_{table_name}"
            tables[key] = df

    for name, df in tables.items():
        print(f"{name}: {df.shape}")


    with sqlite3.connect("example.db", timeout=60) as conn:
        for key, df in tables.items():
            sql_table_name = key

            # append if table exists, create if not
            df.to_sql(
                sql_table_name,
                conn,
                if_exists="append",
                index=False
            )

            # remove duplicate rows from the table
            deduped_df = pd.read_sql_query(
                f'SELECT DISTINCT * FROM "{sql_table_name}"',
                conn
            )

            deduped_df.to_sql(
                sql_table_name,
                conn,
                if_exists="replace",
                index=False
            )

            print(f"{sql_table_name}: {deduped_df.shape}")
            
    return tables

data = get_seasonal_stats('westbru01')
data

2009_player_game_log_reg: (82, 36)
2010_player_game_log_reg: (82, 36)
2010_player_game_log_post: (6, 36)
2011_player_game_log_reg: (82, 36)
2011_player_game_log_post: (17, 36)
2012_player_game_log_reg: (66, 36)
2012_player_game_log_post: (20, 36)
2013_player_game_log_reg: (82, 36)
2013_player_game_log_post: (2, 36)
2014_player_game_log_reg: (46, 36)
2014_player_game_log_post: (19, 36)
2015_player_game_log_reg: (67, 36)
2016_player_game_log_reg: (80, 36)
2016_player_game_log_post: (18, 36)
2017_player_game_log_reg: (81, 36)
2017_player_game_log_post: (5, 36)
2018_player_game_log_reg: (80, 36)
2018_player_game_log_post: (6, 36)
2019_player_game_log_reg: (73, 36)
2019_player_game_log_post: (5, 36)
2020_player_game_log_reg: (57, 36)
2020_player_game_log_post: (8, 36)
2021_player_game_log_reg: (65, 36)
2021_player_game_log_post: (5, 36)
2022_player_game_log_reg: (78, 36)
2023_player_game_log_reg: (73, 36)
2023_player_game_log_post: (5, 36)
2024_player_game_log_reg: (68, 36)
2024_player_game

{'2009_player_game_log_reg':    ranker player_game_num_career team_game_num_season        date  \
 0       1                      1                    1  2008-10-29   
 1       2                      2                    2  2008-11-01   
 2       3                      3                    3  2008-11-02   
 3       4                      4                    4  2008-11-05   
 4       5                      5                    5  2008-11-07   
 ..    ...                    ...                  ...         ...   
 77     78                     78                   78  2009-04-08   
 78     79                     79                   79  2009-04-10   
 79     80                     80                   80  2009-04-11   
 80     81                     81                   81  2009-04-13   
 81     82                     82                   82  2009-04-15   
 
    team_name_abbr game_location opp_name_abbr game_result is_starter     mp  \
 0             OKC          <NA>           MIL    

In [7]:
get_player_data('johnsja05')
get_seasonal_stats('johnsja05')

past_5_games: (5, 28)
per_game_stats: (5, 32)
per_game_stats_post: (3, 32)
stathead_insight: (7, 4)
totals_stats: (5, 33)
totals_stats_post: (3, 33)
per_minute_stats: (5, 32)
per_minute_stats_post: (3, 32)
advanced: (5, 30)
advanced_post: (3, 30)
shooting: (5, 32)
shooting_post: (3, 32)
past_5_games: 5 inserted, 10 total after dedupe
per_game_stats: 5 inserted, 25 total after dedupe
per_game_stats_post: 3 inserted, 17 total after dedupe
stathead_insight: 7 inserted, 14 total after dedupe
totals_stats: 5 inserted, 25 total after dedupe
totals_stats_post: 3 inserted, 17 total after dedupe
per_minute_stats: 5 inserted, 25 total after dedupe
per_minute_stats_post: 3 inserted, 17 total after dedupe
advanced: 5 inserted, 25 total after dedupe
advanced_post: 3 inserted, 17 total after dedupe
shooting: 5 inserted, 25 total after dedupe
shooting_post: 3 inserted, 17 total after dedupe
2022_player_game_log_reg: (22, 36)
2022_player_game_log_post: (2, 35)
2023_player_game_log_reg: (70, 36)
2023_p

{'2022_player_game_log_reg':    ranker player_game_num_career team_game_num_season        date  \
 0       1                      1                    1  2021-10-21   
 2       2                      2                    3  2021-10-25   
 5       3                      3                    6  2021-10-30   
 8       4                      4                    9  2021-11-04   
 13      5                      5                   14  2021-11-14   
 14      6                      6                   15  2021-11-15   
 17      7                      7                   18  2021-11-22   
 29      8                      8                   30  2021-12-22   
 30      9                      9                   31  2021-12-23   
 31     10                     10                   32  2021-12-25   
 38     11                     11                   39  2022-01-09   
 46     12                     12                   47  2022-01-26   
 47     13                     13                   48  2022-0

In [12]:
with sqlite3.connect('example.db') as conn:
    table = pd.read_sql_query(
        'SELECT * FROM "2022_player_game_log_post" WHERE player = "westbru01"',
        conn
    )

table.columns

Index(['ranker', 'player_game_num_career', 'team_game_num_season', 'date',
       'team_name_abbr', 'game_location', 'opp_name_abbr', 'game_result',
       'is_starter', 'mp', 'fg', 'fga', 'fg_pct', 'fg3', 'fg3a', 'fg3_pct',
       'fg2', 'fg2a', 'fg2_pct', 'efg_pct', 'ft', 'fta', 'orb', 'drb', 'trb',
       'ast', 'stl', 'blk', 'tov', 'pf', 'pts', 'game_score', 'plus_minus',
       'player', 'year'],
      dtype='object')